## Load Required Libraries

In [16]:
import pandas as pd
import json
import plotly.graph_objects as go
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm


# ── Load table

In [17]:


# ── Check structure ───────────────────────────────────────────
df = pd.read_csv("hospitals_postcodes.csv")

# Check structure
print("Columns:", df.columns.tolist())
print(f"Rows: {len(df)}")
print(df.head(10))

Columns: ['Code', 'Name', 'Primary_Role_Name', 'Non_Primary_Role_Name(s)', 'Postcode', 'UPRN', 'Geographic_Local_Authority_Code', 'Geographic_Local_Authority_Name', 'High_Level_Health_Geography_Code', 'High_Level_Health_Geography_Name', 'National_Grouping_Code', 'National_Grouping_Name', 'Geographic_Government_Office_Region_Code', 'Geographic_Government_Office_Region_Name']
Rows: 247
  Code                                               Name Primary_Role_Name  \
0  RCF                      AIREDALE NHS FOUNDATION TRUST         NHS TRUST   
1  RBS          ALDER HEY CHILDREN'S NHS FOUNDATION TRUST         NHS TRUST   
2  RTK  ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATIO...         NHS TRUST   
3  RVN  AVON AND WILTSHIRE MENTAL HEALTH PARTNERSHIP N...         NHS TRUST   
4  RF4  BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...         NHS TRUST   
5  RRP  BARNET, ENFIELD AND HARINGEY MENTAL HEALTH NHS...         NHS TRUST   
6  RFF             BARNSLEY HOSPITAL NHS FOUNDATION TRUST   

# ── Geocode ─────────────────────────

In [18]:
import pandas as pd
import requests
import time

# ── Load file ─────────────────────────────────────────────────
df = pd.read_csv("hospitals_postcodes.csv")
df = df[["Code", "Name", "Postcode"]].copy()
df["Postcode"] = df["Postcode"].str.strip().str.upper()

print("Postcodes to geocode:", len(df))
print("\nSample postcodes:")
print(df["Postcode"].head(5))

# ── Geocode using Postcodes.io API ────────────────────────────
# This API is free and doesn't require downloads or SSL certificates
latitudes = []
longitudes = []
geocoded = []

for idx, postcode in enumerate(df["Postcode"]):
    try:
        # Postcodes.io is free and doesn't need API key
        url = f"https://api.postcodes.io/postcodes/{postcode}"
        response = requests.get(url, timeout=5)
        
        if response.status_code == 200:
            data = response.json()
            if data["result"]:
                latitudes.append(data["result"]["latitude"])
                longitudes.append(data["result"]["longitude"])
                geocoded.append(True)
            else:
                latitudes.append(None)
                longitudes.append(None)
                geocoded.append(False)
        else:
            latitudes.append(None)
            longitudes.append(None)
            geocoded.append(False)
        
        # Print progress every 10 records
        if (idx + 1) % 10 == 0:
            print(f"Processed {idx + 1}/{len(df)}")
        
        # Be nice to the API — small delay
        time.sleep(0.1)
        
    except Exception as e:
        print(f"Error for {postcode}: {e}")
        latitudes.append(None)
        longitudes.append(None)
        geocoded.append(False)

df["latitude"] = latitudes
df["longitude"] = longitudes
df["geocoded"] = geocoded

# ── Summary ───────────────────────────────────────────────────
print(f"\nGeocoded:  {sum(geocoded)} / {len(df)}")
print(f"Failed:    {len(df) - sum(geocoded)}")

if not all(geocoded):
    print("\nFailed hospitals:")
    print(df[~df["geocoded"]][["Code", "Name", "Postcode"]].to_string(index=False))

# ── Save ──────────────────────────────────────────────────────
df.to_csv("hospital_coordinates.csv", index=False)
print("\nSaved to hospital_coordinates.csv")
print("\nFirst 10 results:")
print(df[["Code", "Name", "Postcode", "latitude", "longitude"]].head(10).to_string(index=False))


Postcodes to geocode: 247

Sample postcodes:
0    BD20 6TD
1     L12 2AP
2    KT16 0PZ
3     BA1 3QE
4     RM7 0AG
Name: Postcode, dtype: object
Processed 10/247
Processed 10/247
Processed 20/247
Processed 20/247
Processed 30/247
Processed 30/247
Processed 40/247
Processed 40/247
Processed 50/247
Processed 50/247
Processed 60/247
Processed 60/247
Processed 70/247
Processed 70/247
Processed 80/247
Processed 80/247
Processed 90/247
Processed 90/247
Processed 100/247
Processed 100/247
Processed 110/247
Processed 110/247
Processed 120/247
Processed 120/247
Processed 130/247
Processed 130/247
Processed 140/247
Processed 140/247
Processed 150/247
Processed 150/247
Processed 160/247
Processed 160/247
Processed 170/247
Processed 170/247
Processed 180/247
Processed 180/247
Processed 190/247
Processed 190/247
Processed 200/247
Processed 200/247
Processed 210/247
Processed 210/247
Processed 220/247
Processed 220/247
Processed 230/247
Processed 230/247
Processed 240/247
Processed 240/247

Geocoded

# ── Summary ─────────

In [20]:
print(f"\nGeocoded:  {df['geocoded'].sum()} / {len(df)}")
print(f"Failed:    {(~df['geocoded']).sum()}")

print(df.head(10))



Geocoded:  246 / 247
Failed:    1
  Code                                               Name  Postcode  \
0  RCF                      AIREDALE NHS FOUNDATION TRUST  BD20 6TD   
1  RBS          ALDER HEY CHILDREN'S NHS FOUNDATION TRUST   L12 2AP   
2  RTK  ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATIO...  KT16 0PZ   
3  RVN  AVON AND WILTSHIRE MENTAL HEALTH PARTNERSHIP N...   BA1 3QE   
4  RF4  BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...   RM7 0AG   
5  RRP  BARNET, ENFIELD AND HARINGEY MENTAL HEALTH NHS...   N15 3TH   
6  RFF             BARNSLEY HOSPITAL NHS FOUNDATION TRUST   S75 2EP   
7  RNJ                     BARTS AND THE LONDON NHS TRUST    E1 1BB   
8  R1H                             BARTS HEALTH NHS TRUST    E1 2ES   
9  RDD  BASILDON AND THURROCK UNIVERSITY HOSPITALS NHS...  SS16 5NL   

    latitude  longitude  geocoded  
0  53.897957  -1.962729      True  
1  53.421263  -2.897796      True  
2  51.377831  -0.527041      True  
3  51.388601  -2.392311      True  
4  

Overlay with ICB Boundaries

In [22]:
import pandas as pd
import plotly.graph_objects as go
import json

# ── Load files ────────────────────────────────────────────────
df          = pd.read_csv("hospital_coordinates.csv")
icb_geojson = json.load(open("icb_boundaries.geojson", encoding="utf-8"))

# ── Check GeoJSON property names ──────────────────────────────
print("ICB GeoJSON properties:")
print(icb_geojson["features"][0]["properties"])

ICB GeoJSON properties:
{'FID': 1, 'ICB23CD': 'E54000008', 'ICB23NM': 'NHS Cheshire and Merseyside Integrated Care Board', 'BNG_E': 374405, 'BNG_N': 380936, 'LONG': -2.38572, 'LAT': 53.32473, 'GlobalID': 'ad50edea-59a7-4bab-94f5-8b6f4ca04f96'}


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import json

# ── Load files ────────────────────────────────────────────────
df          = pd.read_csv("hospital_coordinates.csv")
icb_geojson = json.load(open("icb_boundaries.geojson", encoding="utf-8"))

# ── Plot ──────────────────────────────────────────────────────
fig = go.Figure()

# ── Layer 1: ICB boundaries ───────────────────────────────────
for feature in icb_geojson["features"]:
    props    = feature["properties"]
    icb_name = props["ICB23NM"]
    icb_code = props["ICB23CD"]
    geo_type = feature["geometry"]["type"]
    coords   = feature["geometry"]["coordinates"]

    # Handle both Polygon and MultiPolygon
    polygons = coords if geo_type == "MultiPolygon" else [coords]

    for polygon in polygons:
        for ring in polygon:
            lons = [pt[0] for pt in ring]
            lats = [pt[1] for pt in ring]

            fig.add_trace(go.Scattermapbox(
                lon=lons,
                lat=lats,
                mode="lines",
                fill="toself",
                fillcolor="rgba(100, 149, 237, 0.15)",  # light blue fill
                line=dict(color="#2c5f8a", width=1),
                text=f"<b>{icb_name}</b><br>Code: {icb_code}",
                hoverinfo="text",
                showlegend=False
            ))

# ── Layer 2: Hospital nodes ───────────────────────────────────
fig.add_trace(go.Scattermapbox(
    lat=df["latitude"],
    lon=df["longitude"],
    mode="markers",
    name="Hospitals",
    marker=dict(
        size=8,
        color="#f97316",
        opacity=0.85,
    ),
    text=(
        "<b>" + df["Name"] + "</b><br>" +
        "Code: "     + df["Code"]     + "<br>" +
        "Postcode: " + df["Postcode"]
    ),
    hoverinfo="text",
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#f97316",
        font=dict(size=11)
    )
))

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=52.5, lon=-1.8),
        zoom=5.8
    ),
    title=dict(
        text=(
            "NHS Hospital Locations across England<br>"
            "<sup>Overlaid on ICB Boundaries</sup>"
        ),
        font=dict(size=16),
        x=0.5
    ),
    legend=dict(
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
        font=dict(size=11)
    ),
    margin=dict(l=0, r=0, t=70, b=0),
    height=800
)

fig.write_html("hospital_icb_map.html")
print("Saved to hospital_icb_map.html")

# Open in browser automatically
import webbrowser
import os
webbrowser.open("file://" + os.path.abspath("hospital_icb_map.html"))



/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_75590/355772473.py:28: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_75590/355772473.py:41: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_75590/355772473.py:41: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


Saved to hospital_icb_map.html


True